# ♻️ Notebook 2: Refactoring a God-Class to SOLID

We'll start from a realistic, deliberately-bad `Order` class and refactor it **one SOLID principle at a time** until we end at production-shape code — complete with unit tests.

## 🛠️ Setup

```bash
cd 07-object-oriented-design/oo-analysis-and-design
uv sync
```

Select the `.venv` kernel in VS Code (top-right). If it doesn't appear, reload the window: `Cmd+Shift+P` → **Reload Window**.

## Step 0 — The god-class (everything in one place)

This is the kind of class a junior engineer writes under deadline pressure. It works… until requirements change.

In [1]:
class BadOrder:
    def __init__(self, items, user_email):
        self.items = items
        self.user_email = user_email

    def total(self):
        return sum(p for _, p in self.items)

    def charge_stripe(self, card_number):
        print(f'  [stripe] charging ${self.total()} on card {card_number}')
        return 'ok'

    def send_confirmation(self):
        print(f'  [smtp] → {self.user_email}: total ${self.total()}')

    def save(self):
        print(f'  [db] INSERT INTO orders ... {self.items}')

o = BadOrder([('book', 10), ('pen', 2)], 'alice@example.com')
o.charge_stripe('4242')
o.send_confirmation()
o.save()

  [stripe] charging $12 on card 4242
  [smtp] → alice@example.com: total $12
  [db] INSERT INTO orders ... [('book', 10), ('pen', 2)]


### What's wrong?

| Principle | Violation |
|-----------|-----------|
| **SRP** | Billing + email + DB + domain logic share one class — four reasons to change. |
| **OCP** | New payment provider? Edit this file. |
| **DIP** | Hardcoded to Stripe and SMTP — you can't unit-test without hitting real services. |
| **Testability** | Every test prints fake data to the console or would call real APIs. |

## Step 1 — Apply SRP: split by responsibility

First, separate the four jobs. `Order` becomes pure data; each side-effect gets its own class.

In [2]:
from dataclasses import dataclass

@dataclass
class Order:
    items: list[tuple[str, float]]
    user_email: str
    def total(self) -> float:
        return sum(p for _, p in self.items)

class StripeBilling:
    def charge(self, amount, token):
        print(f'  [stripe] charged ${amount}')
        return 'ok'

class EmailConfirmations:
    def send(self, to, msg):
        print(f'  [email→{to}] {msg}')

class OrderSqlRepo:
    def save(self, order):
        print(f'  [db] saved order of ${order.total()}')

# The orchestrator — but notice it still HARDCODES concretions
class OrderServiceV1:
    def __init__(self):
        self.billing = StripeBilling()
        self.email = EmailConfirmations()
        self.repo = OrderSqlRepo()

    def place(self, order, token):
        self.billing.charge(order.total(), token)
        self.repo.save(order)
        self.email.send(order.user_email, f'total: ${order.total()}')

OrderServiceV1().place(Order([('book', 10)], 'alice@example.com'), 'tok')

  [stripe] charged $10
  [db] saved order of $10
  [email→alice@example.com] total: $10


✅ **SRP done** — each class has one reason to change.  
❌ **DIP still violated** — `OrderServiceV1` knows about Stripe, SMTP, and SQL by name.

## Step 2 — Apply DIP + OCP: program to interfaces

Define abstract interfaces. Then the service depends only on the *shape* of its collaborators, not on a specific vendor. This also gives us OCP: adding PayPal means adding a class, not editing existing code.

In [3]:
from abc import ABC, abstractmethod

class PaymentGateway(ABC):
    @abstractmethod
    def charge(self, amount: float, token: str) -> str: ...

class Notifier(ABC):
    @abstractmethod
    def send(self, to: str, message: str) -> None: ...

class OrderRepo(ABC):
    @abstractmethod
    def save(self, order: Order) -> None: ...

# Concrete implementations — each can be swapped freely
class StripeGateway(PaymentGateway):
    def charge(self, amount, token):
        print(f'  [stripe] charged ${amount}')
        return 'ok'

class PayPalGateway(PaymentGateway):
    def charge(self, amount, token):
        print(f'  [paypal] charged ${amount}')
        return 'ok'

class EmailNotifier(Notifier):
    def send(self, to, message):
        print(f'  [email→{to}] {message}')

class SmsNotifier(Notifier):
    def send(self, to, message):
        print(f'  [sms→{to}] {message}')

class InMemoryRepo(OrderRepo):
    def __init__(self): self.store = []
    def save(self, order):
        self.store.append(order)
        print(f'  [memory] saved (#{len(self.store)})')

In [4]:
# Service depends on abstractions only — dependencies are INJECTED
class OrderService:
    def __init__(self, repo: OrderRepo, gateway: PaymentGateway, notifier: Notifier):
        self.repo = repo
        self.gateway = gateway
        self.notifier = notifier

    def place(self, order: Order, payment_token: str) -> None:
        self.gateway.charge(order.total(), payment_token)
        self.repo.save(order)
        self.notifier.send(order.user_email, f'Your total: ${order.total()}')

print('--- Stripe + Email ---')
OrderService(InMemoryRepo(), StripeGateway(), EmailNotifier()).place(
    Order([('book', 10), ('pen', 2)], 'alice@example.com'), 'tok_test')

print('--- PayPal + SMS (switched by composition, not code edits) ---')
OrderService(InMemoryRepo(), PayPalGateway(), SmsNotifier()).place(
    Order([('hat', 20)], '+15551234'), 'tok_test')

--- Stripe + Email ---
  [stripe] charged $12
  [memory] saved (#1)
  [email→alice@example.com] Your total: $12
--- PayPal + SMS (switched by composition, not code edits) ---
  [paypal] charged $20
  [memory] saved (#1)
  [sms→+15551234] Your total: $20


## Step 3 — Apply ISP: split fat interfaces when they grow

Imagine later we need refunds. The tempting move is to grow `PaymentGateway` into `charge + refund + tokenize + webhook_verify`. That breaks ISP — now a simple test double must implement 4 methods it doesn't need.

Better: keep small, focused interfaces and let a single class implement multiple.

In [5]:
class Chargeable(ABC):
    @abstractmethod
    def charge(self, amount: float, token: str) -> str: ...

class Refundable(ABC):
    @abstractmethod
    def refund(self, charge_id: str) -> str: ...

class StripeV2(Chargeable, Refundable):
    def charge(self, amount, token):  return f'ch_{amount}'
    def refund(self, charge_id):      return f'refunded {charge_id}'

# A minimal gateway that only supports charging (say, an old cash terminal)
class CashOnlyGateway(Chargeable):
    def charge(self, amount, token): return f'cash_{amount}'

# Refund flow only needs Refundable — CashOnlyGateway is correctly incompatible
def issue_refund(gw: Refundable, charge_id: str):
    return gw.refund(charge_id)

print(issue_refund(StripeV2(), 'ch_42'))

refunded ch_42


## Step 4 — The payoff: fast, isolated unit tests

Because dependencies are injected, the service is testable without Stripe, SMTP, or a real DB.

In [6]:
# Fake doubles that record calls
class FakeGateway(PaymentGateway):
    def __init__(self): self.charges = []
    def charge(self, amount, token):
        self.charges.append((amount, token))
        return 'ok'

class FakeNotifier(Notifier):
    def __init__(self): self.messages = []
    def send(self, to, message):
        self.messages.append((to, message))

class FakeRepo(OrderRepo):
    def __init__(self): self.saved = []
    def save(self, order): self.saved.append(order)

# === 'unit tests' — simple asserts so the notebook is self-checking ===
def test_place_order_happy_path():
    gw, note, repo = FakeGateway(), FakeNotifier(), FakeRepo()
    svc = OrderService(repo, gw, note)

    svc.place(Order([('book', 10), ('pen', 2)], 'alice@example.com'), 'tok_test')

    assert gw.charges == [(12, 'tok_test')],      f'charges wrong: {gw.charges}'
    assert len(repo.saved) == 1,                  'order should be saved once'
    assert note.messages[0][0] == 'alice@example.com'
    assert '$12' in note.messages[0][1]
    print('✅ happy-path test passed')

def test_empty_order_still_charges_zero():
    gw, note, repo = FakeGateway(), FakeNotifier(), FakeRepo()
    OrderService(repo, gw, note).place(Order([], 'bob@example.com'), 'tok')
    assert gw.charges == [(0, 'tok')]
    print('✅ empty-order test passed')

test_place_order_happy_path()
test_empty_order_still_charges_zero()

✅ happy-path test passed
✅ empty-order test passed


## Step 5 — Score card

| Principle | Before | After |
|-----------|--------|-------|
| **SRP** | Everything in `BadOrder` | `Order` data, `OrderService` orchestrates, each side-effect isolated |
| **OCP** | New gateway = edit file | New gateway = new class implementing `PaymentGateway` |
| **LSP** | n/a (no inheritance) | All `PaymentGateway` subclasses interchangeable |
| **ISP** | n/a | `Chargeable` and `Refundable` split so minimal gateways stay minimal |
| **DIP** | `import stripe` everywhere | Service depends only on interfaces; fakes enable fast tests |

This is the **shape most production services take**: a thin orchestrator surrounded by single-purpose adapters, all glued together by dependency injection at the edges (a `main()`, a factory, or a framework's DI container).

## 🧪 Exercises

Try these on your own to cement the pattern:

1. **Add a `LoggingGateway` decorator** that wraps any `PaymentGateway` and prints every call before delegating. This is OCP + the Decorator pattern in one.
2. **Add a `Discount` strategy** — an interface with `apply(order) -> float`. Implement `PercentOff` and `FixedOff`. Inject it into `OrderService`.
3. **Swap `InMemoryRepo` for a `JsonFileRepo`** without touching `OrderService`. If you can, you've internalized DIP.